# 📓 [Day 29 실전 워크북] Cypher 그래프 질의 언어(GQL) 실전 핸즈온

> **학습 목표**:
> 1. `CREATE`와 `MERGE`의 결정적 차이를 이해하고 멱등한 노드/관계를 구축한다.
> 2. `date()`, 리스트 등 다양한 자료형과 다중 레이블(`:A:B`)을 활용한다.
> 3. `SET`, `REMOVE`, `DETACH DELETE`로 그래프를 안전하게 조작한다.
> 4. `WHERE` 조건, 2-Hop 공유 허브 패턴 매칭, `$params` 파라미터 바인딩을 마스터한다.

---

In [ ]:
# [환경 설정] Neo4j 접속 및 헬퍼 함수 정의
import os
from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv(".env", override=True)
load_dotenv("../.env", override=True)
load_dotenv("내작업폴더/day28_Neo4j_설치_Movies/.env", override=True)

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "test0011")

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()

def run_cypher(query: str, **params):
    """Cypher 쿼리 실행 후 dict 리스트로 반환하는 헬퍼"""
    with driver.session() as session:
        return [record.data() for record in session.run(query, **params)]

print("✅ Neo4j 연결 성공:", NEO4J_URI)

In [ ]:
# [초기화] 노바랩스(Nova) 네임스페이스 격리 초기화
run_cypher("MATCH (n:NovaEmployee) DETACH DELETE n")
run_cypher("MATCH (n:NovaTeam) DETACH DELETE n")
run_cypher("MATCH (n:NovaProject) DETACH DELETE n")
print("🧹 노바랩스 실습 그래프 초기화 완료!")

## 🎯 미션 1. 멱등한 노드 생성 (MERGE)
- 팀 노드 2개를 생성하세요:
  1. `NovaTeam {team_id: 'TEAM_AI', name: 'AI Research Lab', room: '701호'}`
  2. `NovaTeam {team_id: 'TEAM_DEV', name: 'Backend Platform', room: '702호'}`

In [ ]:
# [TODO] 미션 1 작성
q1 = """
MERGE (t1:NovaTeam {team_id: 'TEAM_AI', name: 'AI Research Lab', room: '701호'})
MERGE (t2:NovaTeam {team_id: 'TEAM_DEV', name: 'Backend Platform', room: '702호'})
"""
run_cypher(q1)
print("팀 노드 생성 완료!")

In [ ]:
# [자가채점] 미션 1 검증
teams = run_cypher("MATCH (t:NovaTeam) RETURN t.team_id AS id, t.name AS name ORDER BY id")
assert len(teams) == 2, f"Expected 2 teams, got {len(teams)}"
assert teams[0]['id'] == 'TEAM_AI' and teams[1]['id'] == 'TEAM_DEV'
print("✅ [미션 1 통과!] 팀 노드가 멱등하게 생성되었습니다.")

## 🎯 미션 2. 자료형(date, list) 및 다중 레이블 노드 생성
- 직원 2명을 생성하세요:
  1. `Alice`: 레이블 `:NovaEmployee:NovaLead`, `emp_id: 'E001'`, `joined_date: date('2023-01-15')`, `skills: ['Python', 'Neo4j']`, `salary: 8500`
  2. `Bob`: 레이블 `:NovaEmployee`, `emp_id: 'E002'`, `joined_date: date('2024-03-01')`, `skills: ['Python', 'Docker']`, `salary: 7000`

In [ ]:
# [TODO] 미션 2 작성
q2 = """
MERGE (e1:NovaEmployee:NovaLead {emp_id: 'E001'})
ON CREATE SET 
    e1.name = 'Alice',
    e1.joined_date = date('2023-01-15'),
    e1.skills = ['Python', 'Neo4j'],
    e1.salary = 8500

MERGE (e2:NovaEmployee {emp_id: 'E002'})
ON CREATE SET 
    e2.name = 'Bob',
    e2.joined_date = date('2024-03-01'),
    e2.skills = ['Python', 'Docker'],
    e2.salary = 7000
"""
run_cypher(q2)
print("직원 노드 생성 완료!")

In [ ]:
# [자가채점] 미션 2 검증
alice = run_cypher("MATCH (e:NovaEmployee {emp_id: 'E001'}) RETURN e.name AS name, labels(e) AS lbls, e.skills AS skills")[0]
assert 'NovaLead' in alice['lbls'], "Alice does not have NovaLead label"
assert 'Neo4j' in alice['skills'], "Alice does not have Neo4j skill"
print("✅ [미션 2 통과!] 다중 레이블 및 리스트 속성이 완벽하게 반영되었습니다.")

## 🎯 미션 3. 관계선 연결 및 관계 속성 (WORKS_IN)
- `Alice(E001)`와 `Bob(E002)`를 `AI Research Lab(TEAM_AI)`에 `:WORKS_IN` 관계로 연결하세요.
  - Alice의 관계 속성: `since: 2023`
  - Bob의 관계 속성: `since: 2024`

In [ ]:
# [TODO] 미션 3 작성
q3 = """
MATCH (e1:NovaEmployee {emp_id: 'E001'}), (t:NovaTeam {team_id: 'TEAM_AI'})
MERGE (e1)-[:WORKS_IN {since: 2023}]->(t)

MATCH (e2:NovaEmployee {emp_id: 'E002'}), (t:NovaTeam {team_id: 'TEAM_AI'})
MERGE (e2)-[:WORKS_IN {since: 2024}]->(t)
"""
run_cypher(q3)
print("관계선 연결 완료!")

In [ ]:
# [자가채점] 미션 3 검증
members = run_cypher("MATCH (e:NovaEmployee)-[r:WORKS_IN]->(t:NovaTeam {team_id: 'TEAM_AI'}) RETURN e.name AS name, r.since AS since ORDER BY name")
assert len(members) == 2, f"Expected 2 members, got {len(members)}"
assert members[0]['name'] == 'Alice' and members[0]['since'] == 2023
print("✅ [미션 3 통과!] 팀 소속 관계선 및 관계 속성이 정상 등록되었습니다.")

## 🎯 미션 4. 수정(SET) 및 레이블 승격
- `Bob(E002)`의 연봉(`salary`)을 7800으로 올리고, `:NovaLead` 레이블을 추가하세요.

In [ ]:
# [TODO] 미션 4 작성
q4 = """
MATCH (e:NovaEmployee {emp_id: 'E002'})
SET e:NovaLead, e.salary = 7800, e.promoted_at = date()
"""
run_cypher(q4)
print("Bob 승진 및 연봉 수정 완료!")

In [ ]:
# [자가채점] 미션 4 검증
bob = run_cypher("MATCH (e:NovaEmployee {emp_id: 'E002'}) RETURN e.salary AS sal, labels(e) AS lbls")[0]
assert bob['sal'] == 7800, "Bob salary is not 7800"
assert 'NovaLead' in bob['lbls'], "Bob does not have NovaLead label"
print("✅ [미션 4 통과!] SET을 통한 속성 변경 및 레이블 동적 부여 성공!")

## 🎯 미션 5. 2-Hop 공유 허브 질의 (같은 팀 동료 찾기)
- `Alice`와 같은 팀에 속한 동료의 이름을 2-Hop 패턴(`(Alice)-[:WORKS_IN]->(Team)<-[:WORKS_IN]-(Colleague)`)으로 조회하세요.

In [ ]:
# [TODO] 미션 5 작성
q5 = """
MATCH (me:NovaEmployee {name: 'Alice'})-[:WORKS_IN]->(t:NovaTeam)<-[:WORKS_IN]-(colleague:NovaEmployee)
RETURN colleague.name AS colleague_name, t.name AS team_name
"""
colleagues = run_cypher(q5)
print("2-Hop 동료 조회 결과:", colleagues)

In [ ]:
# [자가채점] 미션 5 검증
assert len(colleagues) == 1, f"Expected 1 colleague, got {len(colleagues)}"
assert colleagues[0]['colleague_name'] == 'Bob'
print("✅ [미션 5 통과!] 2-Hop 인덱스 프리 인접성(IFA) 동료 추론 성공!")

## 🎯 미션 6. 파라미터($params) 바인딩 질의
- 최소 연봉 `$min_sal`을 파라미터로 넘겨 연봉이 `$min_sal` 이상인 직원의 이름과 연봉을 내림차순 정렬하여 반환하세요.

In [ ]:
# [TODO] 미션 6 작성
q6 = """
MATCH (e:NovaEmployee)
WHERE e.salary >= $min_sal
RETURN e.name AS name, e.salary AS salary
ORDER BY e.salary DESC
"""
high_earners = run_cypher(q6, min_sal=7500)
print("파라미터 질의 결과:", high_earners)

In [ ]:
# [자가채점] 미션 6 검증
assert len(high_earners) == 2, f"Expected 2 rows, got {len(high_earners)}"
assert high_earners[0]['name'] == 'Alice' and high_earners[0]['salary'] == 8500
assert high_earners[1]['name'] == 'Bob' and high_earners[1]['salary'] == 7800
print("🎉 [미션 6 통과!] Day 29 Cypher 실전 워크북의 모든 미션을 100% 완주하셨습니다!")